# Exercise: Implementing a PagedAttention Block Allocator

Welcome to the exercise on PagedAttention! In the preceding material, you learned how PagedAttention avoids internal fragmentation by managing the KV cache in non-contiguous blocks, much like virtual memory paging in an operating system.

The heart of this system is the `BlockAllocator`, which is responsible for managing the pool of available memory blocks. In this exercise, you will implement a simplified `BlockAllocator` from scratch.

**You will learn to:**
*   Initialize an allocator with a pool of free blocks.
*   Implement methods to `allocate`, `extend`, and `free` blocks for different requests.
*   Track the mapping from a request ID to its assigned blocks using a block table.

Let's get started!

In [ ]:
import collections
from typing import List, Dict, Set

# --- Type Aliases for clarity ---
BlockId = int
RequestId = int

class BlockAllocator:
    """
    Manages a pool of KV cache blocks for PagedAttention.

    This is a simplified implementation that tracks free blocks and allocates
    them to requests.
    """
    def __init__(self, num_blocks: int):
        """
        Initializes the BlockAllocator.

        Args:
            num_blocks: The total number of physical blocks available in the system.
        """
        self.num_blocks = num_blocks
        self.free_blocks: List[BlockId] = []
        self.block_tables: Dict[RequestId, List[BlockId]] = {}
        # Students will implement the logic to initialize free_blocks

    def allocate(self, request_id: RequestId) -> BlockId:
        """
        Allocates the first block for a new request.

        Args:
            request_id: The ID of the new request.

        Returns:
            The BlockId of the first block allocated to this request.

        Raises:
            ValueError: If no blocks are free.
            KeyError: If the request_id already has allocated blocks.
        """
        # Students will implement this method
        raise NotImplementedError

    def extend(self, request_id: RequestId) -> BlockId:
        """
        Allocates a new block and adds it to an existing request's block table.

        Args:
            request_id: The ID of the request that is growing.

        Returns:
            The BlockId of the newly allocated block.

        Raises:
            ValueError: If no blocks are free.
            KeyError: If the request_id does not exist.
        """
        # Students will implement this method
        raise NotImplementedError

    def free(self, request_id: RequestId) -> None:
        """
        Frees all blocks associated with a completed request.

        Args:
            request_id: The ID of the request to free.

        Raises:
            KeyError: If the request_id does not exist.
        """
        # Students will implement this method
        raise NotImplementedError

    def get_num_free_blocks(self) -> int:
        """Returns the number of free blocks."""
        return len(self.free_blocks)

    def get_block_table(self, request_id: RequestId) -> List[BlockId]:
        """Returns the block table for a given request."""
        return self.block_tables.get(request_id, [])

### Exercise 1: Initializing the Allocator and Allocating New Requests

Your first task is to implement the `__init__` and `allocate` methods.

1.  **`__init__(self, num_blocks)`**: When the allocator is created, it needs a pool of blocks to manage. You should initialize the `self.free_blocks` list to contain all possible block IDs, from `0` to `num_blocks - 1`. A simple way to do this is with `list(range(num_blocks))`. We will treat this list like a stack, where we can `pop` blocks to allocate them and `append` them back when they are freed.

2.  **`allocate(self, request_id)`**: This method is called when a new request arrives. It should:
    *   Check if another request with the same `request_id` already exists. If so, raise a `KeyError`.
    *   Check if there are any free blocks left. If not, raise a `ValueError` with a helpful message like "Out of memory!".
    *   If blocks are available, `pop` one from the `self.free_blocks` list.
    *   Create a new entry in `self.block_tables` for the `request_id`. The value should be a list containing the single block ID you just popped.
    *   Return the allocated block ID.

**Hint**: Using `list.pop()` removes and returns the *last* item from a list, which is an efficient O(1) operation.

In [ ]:
def BlockAllocator_init_and_allocate(BlockAllocator):
    """
    This function returns a modified BlockAllocator class with the
    __init__ and allocate methods implemented.
    """
    class AllocatorWithInit(BlockAllocator):
        def __init__(self, num_blocks: int):
            """
            Initializes the BlockAllocator.

            Args:
                num_blocks: The total number of physical blocks available in the system.
            """
            self.num_blocks = num_blocks
            self.block_tables: Dict[RequestId, List[BlockId]] = {}
    class AllocatorWithInit(BlockAllocator):
        def __init__(self, num_blocks: int):
            """
            Initializes the BlockAllocator.

            Args:
                num_blocks: The total number of physical blocks available in the system.
            """
            self.num_blocks = num_blocks
            self.block_tables: Dict[RequestId, List[BlockId]] = {}
            # Initialize free_blocks with all block IDs from 0 to num_blocks - 1
            self.free_blocks: List[BlockId] = list(range(num_blocks))

        def allocate(self, request_id: RequestId) -> BlockId:
            """
            Allocates the first block for a new request.
            """
            if request_id in self.block_tables:
                raise KeyError(f"Request ID {request_id} already exists.")

            if not self.free_blocks:
                raise ValueError("Out of memory! No free blocks available.")

            # Get a block from the free pool
            block_id = self.free_blocks.pop()

            # Assign it to the new request
            self.block_tables[request_id] = [block_id]

            return block_id

    return AllocatorWithInit

        def allocate(self, request_id: RequestId) -> BlockId:
            """
            Allocates the first block for a new request.
            """
            ### START CODE HERE ###
            if request_id in self.block_tables:
                raise KeyError(f"Request ID {request_id} already exists.")

            if not self.free_blocks:
                raise ValueError("Out of memory! No free blocks available.")

            # Get a block from the free pool
            block_id = self.free_blocks.pop()

            # Assign it to the new request
            self.block_tables[request_id] = [block_id]

            return block_id
            ### END CODE HERE ###

    return AllocatorWithInit

# We patch the original class with your implementation for testing
ImplementedBlockAllocator1 = BlockAllocator_init_and_allocate(BlockAllocator)

In [ ]:
# --- Test Cell for Exercise 1 ---

# Test 1: Initialization
allocator = ImplementedBlockAllocator1(num_blocks=8)
print("Testing initialization...")
assert allocator.get_num_free_blocks() == 8, f"Expected 8 free blocks, got {allocator.get_num_free_blocks()}"
assert allocator.free_blocks == list(range(8)), "free_blocks list not initialized correctly."
assert not allocator.block_tables, "block_tables should be empty on initialization."
print("✅ Initialization tests passed!")

# Test 2: Basic allocation
print("\nTesting basic allocation...")
req1_id = 101
block1 = allocator.allocate(req1_id)
assert allocator.get_num_free_blocks() == 7, f"Expected 7 free blocks after one allocation, got {allocator.get_num_free_blocks()}"
assert block1 not in allocator.free_blocks, f"Allocated block {block1} should not be in free_blocks."
assert allocator.get_block_table(req1_id) == [block1], f"Block table for request {req1_id} is incorrect."
print("✅ Basic allocation test passed!")

# Test 3: Allocating to an existing request ID should fail
print("\nTesting duplicate request ID allocation...")
try:
    allocator.allocate(req1_id)
    # This line should not be reached
    assert False, "Allocating to an existing request_id should raise a KeyError."
except KeyError:
    print("✅ Correctly raised KeyError for duplicate request ID.")

# Test 4: Out of memory
print("\nTesting out of memory condition...")
allocator_small = ImplementedBlockAllocator1(num_blocks=1)
block_a = allocator_small.allocate(1)
try:
    allocator_small.allocate(2)
    assert False, "Allocating when no blocks are free should raise a ValueError."
except ValueError as e:
    print(f"✅ Correctly raised ValueError: {e}")

print("\nAll tests for Exercise 1 passed! Great job!")

### Exercise 2: Extending a Request's Blocks

Fantastic! Your allocator can now handle new requests. But what happens when a request needs more tokens, and therefore more KV cache space? You need to *extend* its block table.

Your task is to implement the `extend` method. It is similar to `allocate` but works on an *existing* request.

**`extend(self, request_id)`**:
*   Check if the `request_id` exists in `self.block_tables`. If it doesn't, this is an invalid request to extend, so you should raise a `KeyError`.
*   Check if there are any free blocks. If not, raise a `ValueError` ("Out of memory!").
*   If checks pass, `pop` a new block from `self.free_blocks`.
*   `append` this new block ID to the list associated with `request_id` in `self.block_tables`.
*   Return the newly allocated block ID.

In [ ]:
def BlockAllocator_extend(BlockAllocatorBase):
    """
    Returns a class that inherits from BlockAllocatorBase and implements the extend method.
    """
    class AllocatorWithExtend(BlockAllocatorBase):
        def extend(self, request_id: RequestId) -> BlockId:
            """
            Allocates a new block and adds it to an existing request's block table.
            """
    class AllocatorWithExtend(BlockAllocatorBase):
        def extend(self, request_id: RequestId) -> BlockId:
            """
            Allocates a new block and adds it to an existing request's block table.
            """
            if request_id not in self.block_tables:
                raise KeyError(f"Request ID {request_id} not found.")

            if not self.free_blocks:
                raise ValueError("Out of memory! Cannot extend.")

            # Get a new block from the free pool
            new_block_id = self.free_blocks.pop()

            # Append it to the existing request's block table
            self.block_tables[request_id].append(new_block_id)

            return new_block_id

    return AllocatorWithExtend

    return AllocatorWithExtend

# We add your new implementation to the class from the previous exercise
ImplementedBlockAllocator2 = BlockAllocator_extend(ImplementedBlockAllocator1)

In [ ]:
# --- Test Cell for Exercise 2 ---
allocator = ImplementedBlockAllocator2(num_blocks=10)
req1_id = 201

# First, allocate a block to have something to extend
initial_block = allocator.allocate(req1_id)
print("State before extend:")
print(f"  Free blocks: {allocator.get_num_free_blocks()}")
print(f"  Block table for {req1_id}: {allocator.get_block_table(req1_id)}")

# Test 1: Basic extend
print("\nTesting basic extend...")
extended_block = allocator.extend(req1_id)
assert allocator.get_num_free_blocks() == 8, f"Expected 8 free blocks, got {allocator.get_num_free_blocks()}"
assert allocator.get_block_table(req1_id) == [initial_block, extended_block], "Block table was not extended correctly."
print(f"State after extend:")
print(f"  Free blocks: {allocator.get_num_free_blocks()}")
print(f"  Block table for {req1_id}: {allocator.get_block_table(req1_id)}")
print("✅ Basic extend test passed!")

# Test 2: Extending a non-existent request
print("\nTesting extend on a non-existent request...")
try:
    allocator.extend(999) # 999 is not a valid request ID
    assert False, "Extending a non-existent request should raise a KeyError."
except KeyError:
    print("✅ Correctly raised KeyError for non-existent request.")

# Test 3: Extending when out of memory
print("\nTesting extend when out of memory...")
allocator_small = ImplementedBlockAllocator2(num_blocks=2)
allocator_small.allocate(1)
allocator_small.allocate(2) # All blocks are now used
assert allocator_small.get_num_free_blocks() == 0
try:
    allocator_small.extend(1)
    assert False, "Extending when no blocks are free should raise a ValueError."
except ValueError as e:
    print(f"✅ Correctly raised ValueError: {e}")

print("\nAll tests for Exercise 2 passed! You're doing great!")

### Exercise 3: Freeing a Request's Blocks

The final piece of the puzzle is memory deallocation. When a request is finished (e.g., it has generated an EOS token or reached its max length), its resources must be returned to the system for others to use. This is what the `free` method does.

**`free(self, request_id)`**:
*   Check if the `request_id` exists. If not, raise a `KeyError`.
*   Retrieve the list of block IDs from `self.block_tables` for this request.
*   Add all of those blocks back to the `self.free_blocks` list. `list.extend()` is perfect for this.
*   Finally, remove the request's entry from `self.block_tables` entirely. The `del` keyword is what you need here.

This completes the lifecycle of a request's memory!

In [ ]:
def BlockAllocator_free(BlockAllocatorBase):
    """
    Returns a class that inherits from BlockAllocatorBase and implements the free method.
    """
    class AllocatorWithFree(BlockAllocatorBase):
        def free(self, request_id: RequestId) -> None:
            """
            Frees all blocks associated with a completed request.
            """
    class AllocatorWithFree(BlockAllocatorBase):
        def free(self, request_id: RequestId) -> None:
            """
            Frees all blocks associated with a completed request.
            """
            if request_id not in self.block_tables:
                raise KeyError(f"Request ID {request_id} not found.")

            # Get the blocks to be freed
            blocks_to_free = self.block_tables[request_id]

            # Add them back to the free pool
            self.free_blocks.extend(blocks_to_free)

            # Remove the request from the active tables
            del self.block_tables[request_id]

    return AllocatorWithFree

    return AllocatorWithFree

# Create the final, fully-implemented BlockAllocator class
BlockAllocator = BlockAllocator_free(ImplementedBlockAllocator2)

In [ ]:
# --- Test Cell for Exercise 3 ---
allocator = BlockAllocator(num_blocks=5)
req1_id = 301

# Setup: Allocate and extend a request so it has multiple blocks
allocator.allocate(req1_id)
allocator.extend(req1_id)
allocator.extend(req1_id)

print(f"State before freeing request {req1_id}:")
print(f"  Free blocks count: {allocator.get_num_free_blocks()}")
print(f"  Allocated blocks for {req1_id}: {allocator.get_block_table(req1_id)}")
print(f"  Active requests: {list(allocator.block_tables.keys())}")
assert allocator.get_num_free_blocks() == 2
assert len(allocator.get_block_table(req1_id)) == 3

# Test 1: Free the request
print(f"\nFreeing request {req1_id}...")
allocator.free(req1_id)

print(f"\nState after freeing request {req1_id}:")
print(f"  Free blocks count: {allocator.get_num_free_blocks()}")
print(f"  Allocated blocks for {req1_id}: {allocator.get_block_table(req1_id)}")
print(f"  Active requests: {list(allocator.block_tables.keys())}")

assert allocator.get_num_free_blocks() == 5, f"Expected all 5 blocks to be free, but got {allocator.get_num_free_blocks()}"
assert not allocator.get_block_table(req1_id), "Block table for freed request should be empty."
assert req1_id not in allocator.block_tables, "Request ID should be removed from block_tables."
print("\n✅ Basic free test passed!")

# Test 2: Freeing a non-existent request
print("\nTesting free on a non-existent request...")
try:
    allocator.free(999) # 999 is not a valid request ID
    assert False, "Freeing a non-existent request should raise a KeyError."
except KeyError:
    print("✅ Correctly raised KeyError for non-existent request.")


print("\nAll tests for Exercise 3 passed! Congratulations on building a complete Block Allocator!")

### (Optional) Challenge: Simulating Request Lifecycles

Now let's put it all together in a small simulation. This will test how your allocator handles a sequence of allocations, extensions, and deallocations, which is a more realistic scenario for an LLM inference server.

The test cell below simulates the following timeline:
1.  Request 1 arrives.
2.  Request 2 arrives.
3.  Request 1 generates a new token (needs an extension).
4.  Request 2 finishes and is freed.
5.  Request 3 arrives and reuses the blocks freed by Request 2.
6.  Request 1 generates another token.

Follow the code and asserts to see how the state of your allocator changes over time. If all your previous implementations are correct, this test should pass without any changes!

In [ ]:
# --- Integration Test: Simulating Multiple Requests ---
print("--- Starting Request Lifecycle Simulation ---")

# Initial state: 10 blocks available
allocator = BlockAllocator(num_blocks=10)
print(f"Initial state: {allocator.get_num_free_blocks()} free blocks.")
assert allocator.get_num_free_blocks() == 10

# 1. Request 101 arrives
req1 = 101
allocator.allocate(req1)
print(f"\nRequest {req1} arrived. Free blocks: {allocator.get_num_free_blocks()}")
assert allocator.get_num_free_blocks() == 9
assert len(allocator.get_block_table(req1)) == 1

# 2. Request 102 arrives
req2 = 102
allocator.allocate(req2)
print(f"Request {req2} arrived. Free blocks: {allocator.get_num_free_blocks()}")
assert allocator.get_num_free_blocks() == 8
assert len(allocator.get_block_table(req2)) == 1

# 3. Request 101 generates a token (extends)
allocator.extend(req1)
print(f"Request {req1} extended. Free blocks: {allocator.get_num_free_blocks()}")
assert allocator.get_num_free_blocks() == 7
assert len(allocator.get_block_table(req1)) == 2

# 4. Request 102 finishes (is freed)
allocator.free(req2)
print(f"Request {req2} freed. Free blocks: {allocator.get_num_free_blocks()}")
assert allocator.get_num_free_blocks() == 8 # 7 + 1 block from req2
assert req2 not in allocator.block_tables

# 5. Request 103 arrives
req3 = 103
allocator.allocate(req3)
print(f"Request {req3} arrived. Free blocks: {allocator.get_num_free_blocks()}")
assert allocator.get_num_free_blocks() == 7 # Reused a block from req2
assert len(allocator.get_block_table(req3)) == 1

# 6. Request 101 generates another token
allocator.extend(req1)
print(f"Request {req1} extended again. Free blocks: {allocator.get_num_free_blocks()}")
assert allocator.get_num_free_blocks() == 6
assert len(allocator.get_block_table(req1)) == 3

# Final state check
print("\n--- Simulation Complete ---")
print("Final state:")
print(f"  Free blocks: {allocator.get_num_free_blocks()}")
print(f"  Active requests: {list(allocator.block_tables.keys())}")
print(f"  Block table for {req1}: {allocator.get_block_table(req1)}")
print(f"  Block table for {req3}: {allocator.get_block_table(req3)}")

num_allocated_blocks = sum(len(table) for table in allocator.block_tables.values())
total_blocks_check = allocator.get_num_free_blocks() + num_allocated_blocks
assert total_blocks_check == allocator.num_blocks, "Memory leak! Total blocks do not match initial number."

print("\n✅ Congratulations! Your BlockAllocator handled the simulation perfectly.")